# Max pooling is a gate

Companion to the Oct 29 lecture *Max pooling is a gate*. It reproduces every computed number and plot on the slides:

1. variance through a conv layer, forward and backward, and why padding lowers it at the edges;
2. a conv written as a linear layer with tied weights;
3. variance through max and average pooling, including where max pooling's 1.03 and 0.49 come from;
4. variance through Tuesday's whole network, forward and backward, at initialization;
5. max pooling as a gate on Tuesday's 7, and the temperature dial from max to average;
6. what happens to training when the pools in Tuesday's network are swapped for soft or learned gates.

**Running it.** MNIST downloads in the first cells. Sections 1–5 take seconds on a CPU. Section 6 trains 18 networks: about 10–15 minutes on a Colab T4 (Runtime → Change runtime type → T4 GPU). Set `QUICK = True` for a two-minute smoke test of section 6. Section 6 caches each run, so an interrupted session picks up where it stopped.

Every number comes from the code below, with the same seeds and settings the slides used. The attention slides on the board's hidden page (next lecture) are not covered here.

In [ ]:
QUICK = False          # True: section 6 runs 1 seed, 1 epoch, 2,000 training digits (smoke test)
USE_DRIVE = False      # True on Colab: keep section 6's results cache in Google Drive

import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from scipy import integrate, stats

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

BLUE, ORANGE, INK, GREY = "#2557D6", "#C2510F", "#16181D", "#8E94A0"

## MNIST

We keep the raw training and test images (values 0–1) on the CPU. Each section standardizes them the way its slide did.

In [ ]:
def load_mnist(train):
    """MNIST images as an N x 1 x 28 x 28 float tensor in [0, 1], and their labels."""
    dataset = torchvision.datasets.MNIST("data", train=train, download=True)
    return dataset.data.float().unsqueeze(1) / 255.0, dataset.targets


images_train, labels_train = load_mnist(train=True)
images_test, labels_test = load_mnist(train=False)
print("train", tuple(images_train.shape), " test", tuple(images_test.shape))

## 1. Variance through a conv layer: forward, and backward

One conv from block 1 of Tuesday's network: GroupNorm output x̂ ~ N(0, 1), then p̂ = ReLU(x̂), then a 3 × 3 conv from 32 to 32 channels on 14 × 14 maps, with He initialization σ_W² = 2 / fan-in and fan-in = C_in · k² = 288.

Going forward we measure each factor of σ²_ŷ = k²C_in · E[w²] · E[p̂²]. Going backward we send an upstream gradient ∂L/∂ŷ of variance 1 and measure it after the conv and after the ReLU. We measure on interior positions only; section 2 shows why the border is different.

In [ ]:
def conv_variance(n_examples=256, channels=32, size=14, k=3, seed=0):
    """Measure every factor of the forward and backward variance rule for one He-initialized 3 x 3 conv."""
    torch.manual_seed(seed)
    fan_in = k * k * channels
    W = torch.randn(channels, channels, k, k) * (2 / fan_in) ** 0.5
    x_hat = torch.randn(n_examples, channels, size, size)        # GroupNorm output: mean 0, variance 1
    p_hat = F.relu(x_hat).requires_grad_(True)
    y_hat = F.conv2d(p_hat, W, padding=1)
    grad_y_hat = torch.randn_like(y_hat)                         # upstream dL/dy_hat, variance 1
    y_hat.backward(grad_y_hat)
    interior = (slice(None), slice(None), slice(1, size - 1), slice(1, size - 1))
    grad_x_hat = p_hat.grad * (x_hat > 0)                        # back through the ReLU: times H(x_hat)
    return {
        "fan-in": fan_in,
        "Var[x_hat]": x_hat.var().item(),
        "E[p_hat^2]": p_hat.detach().pow(2).mean().item(),
        "mean of p_hat": p_hat.detach().mean().item(),
        "Var[p_hat]": p_hat.detach().var().item(),
        "E[w^2]": W.pow(2).mean().item(),
        "Var[y_hat]": y_hat.detach()[interior].var().item(),
        "Var[dL/dp_hat]": p_hat.grad[interior].var().item(),
        "Var[dL/dx_hat]": grad_x_hat[interior].var().item(),
    }


conv_stats = conv_variance()
for name, value in conv_stats.items():
    print(f"{name:16s} {value:8.4f}")
print()
print("forward prediction : 288 * (2/288) * 1/2 =", 288 * (2 / 288) * 0.5)
print("backward prediction: 288 * (2/288) * 1   =", 288 * (2 / 288) * 1.0, "after the conv, then half that after the ReLU")

## 1, continued. Padding shrinks the variance at the edges

Zero padding puts zeros under part of the filter at the border, so fewer real terms enter the sum. We measure σ²_ŷ at every one of the 14 × 14 positions, averaged over 16 independent weight draws so that no single filter favours one border. The last line tries the correction from the slide: scale each output by (k² / N)^½, where N is the number of real inputs under the filter.

In [ ]:
def padding_variance(n_examples=2048, channels=32, size=14, k=3, n_weight_draws=16, seed=0):
    """Per-position variance of a padded, He-initialized 3 x 3 conv; also the number of real inputs per window."""
    torch.manual_seed(seed)
    outputs = []
    for _ in range(n_weight_draws):
        p_hat = F.relu(torch.randn(n_examples // n_weight_draws, channels, size, size))
        W = torch.randn(channels, channels, k, k) * (2 / (k * k * channels)) ** 0.5
        outputs.append(F.conv2d(p_hat, W, padding=1))
    y_hat = torch.cat(outputs)
    real_inputs = F.conv2d(torch.ones(1, 1, size, size), torch.ones(1, 1, k, k), padding=1)[0, 0]
    variance_map = y_hat.var(dim=(0, 1))
    corrected_map = (y_hat * (k * k / real_inputs).sqrt()).var(dim=(0, 1))
    return variance_map, corrected_map, real_inputs


variance_map, corrected_map, real_inputs = padding_variance()
print(f"interior {variance_map[7, 7]:.2f}   edge {variance_map[0, 7]:.2f}   corner {variance_map[0, 0]:.2f}")
print("prediction: interior 9/9 = 1.00, edge 6/9 = 0.67, corner 4/9 = 0.44")
print(f"with the correction: interior {corrected_map[7, 7]:.2f}   edge {corrected_map[0, 7]:.2f}   corner {corrected_map[0, 0]:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, image, title in zip(axes, [variance_map, corrected_map], ["Zero padding", "Scaled by (k²/N)^½"]):
    shown = ax.imshow(image.numpy(), cmap="RdBu_r", vmin=0.4, vmax=1.6)
    ax.set_title(f"{title}: σ²_ŷ per position")
    ax.set_xlabel("column")
    ax.set_ylabel("row")
fig.colorbar(shown, ax=axes, label="σ²_ŷ")
plt.show()

## 2. A conv is a local linear layer with tied weights

The slides unroll a 1-D, one-channel conv with filter (w₁, w₂, w₃) on 8 positions into an 8 × 8 matrix. Here we build that matrix and check two things:

- the forward pass ŷ = p̂ Wᵀ gives the conv's output;
- the gradient of each filter weight is the sum of ∂L/∂W over every matrix entry that holds a copy of it.

There are no slide numbers to match here, only the identities.

In [ ]:
def conv_matrix(filter_weights, length):
    """The length x length matrix W with y_hat = p_hat W^T for a 1-D conv with zero padding 1."""
    W = torch.zeros(length, length)
    for row in range(length):
        for offset, weight in zip([-1, 0, 1], filter_weights):
            column = row + offset
            if 0 <= column < length:
                W[row, column] = weight
    return W


torch.manual_seed(0)
filter_weights = torch.randn(3, requires_grad=True)
p_hat = torch.randn(1, 8)

y_hat_conv = F.conv1d(p_hat.view(1, 1, 8), filter_weights.view(1, 1, 3), padding=1).view(1, 8)
W = conv_matrix(filter_weights.detach(), 8).requires_grad_(True)
y_hat_matrix = p_hat @ W.T
print("forward: conv and matrix agree:", torch.allclose(y_hat_conv, y_hat_matrix))

grad_y_hat = torch.randn(1, 8)                      # an upstream gradient g
y_hat_conv.backward(grad_y_hat)
y_hat_matrix.backward(grad_y_hat)
# dL/dW = g^T p_hat; each filter weight sits on one diagonal of W, so add that diagonal up
summed = torch.stack([W.grad.diagonal(offset).sum() for offset in [-1, 0, 1]])
print("dL/dw from the conv          :", filter_weights.grad.numpy().round(4))
print("dL/dW summed along diagonals :", summed.numpy().round(4))

## 3. Variance through a pool: forward, and backward

2 × 2 windows with stride 2 on inputs with mean 0 and variance 1, then an upstream gradient of variance 1 on each output. The slides predict:

- average pooling: σ² = 4 · (¼)² = ¼ forward, and E[(∂L/∂x)²] = (¼)² = 1/16 backward;
- max pooling: mean 1.03 and σ² 0.49 forward; backward, a quarter of the inputs get the full gradient, so E[(∂L/∂x)²] = ¼.

In [ ]:
def pool_variance(pool, n_examples=256, channels=32, size=14, seed=0):
    """Forward moments of a 2 x 2 pool on N(0, 1) inputs, and the backward second moment of its input gradient."""
    torch.manual_seed(seed)
    x = torch.randn(n_examples, channels, size, size, requires_grad=True)
    out = pool(x, 2)
    out.backward(torch.randn_like(out))
    return {
        "mean out": out.mean().item(),
        "Var out": out.var().item(),
        "E[(dL/dx)^2]": x.grad.pow(2).mean().item(),
        "inputs with a gradient": (x.grad != 0).float().mean().item(),
    }


for name, pool in [("max pool", F.max_pool2d), ("average pool", F.avg_pool2d)]:
    result = pool_variance(pool)
    print(f"{name:13s}", "   ".join(f"{key} {value:.3f}" for key, value in result.items()))

### Where the max pool's 0.49 comes from

The max of 4 independent standard normals is at most x exactly when all four are, so P(max ≤ x) = Φ(x)⁴ and its density is f(x) = 4Φ(x)³φ(x). We integrate for the mean and variance and redraw the slide's plot.

In [ ]:
def density_of_max(x, n=4):
    """Density of the largest of n independent standard normals: n * Phi(x)^(n-1) * phi(x)."""
    return n * stats.norm.cdf(x) ** (n - 1) * stats.norm.pdf(x)


mean_of_max = integrate.quad(lambda x: x * density_of_max(x), -np.inf, np.inf)[0]
var_of_max = integrate.quad(lambda x: (x - mean_of_max) ** 2 * density_of_max(x), -np.inf, np.inf)[0]
print(f"max of 4: mean {mean_of_max:.2f}, variance {var_of_max:.2f}")
print(f"at x = 1: Phi(1) = {stats.norm.cdf(1):.2f}, Phi(1)^4 = {stats.norm.cdf(1) ** 4:.2f}")

x = np.linspace(-3, 3, 241)
plt.figure(figsize=(6, 3.8))
plt.plot(x, stats.norm.cdf(x), color=GREY, linestyle="--", label="one value: Φ(x)")
plt.plot(x, stats.norm.cdf(x) ** 4, color=ORANGE, linewidth=2.5, label="max of 4: Φ(x)⁴")
plt.axvline(1, color=INK, linewidth=0.8, linestyle=":")
plt.title("P(value ≤ x)")
plt.xlabel("x")
plt.ylabel("cumulative probability")
plt.legend()
plt.show()

## 4. Variance through the whole network: forward, and backward

Tuesday's CNN at initialization: a 5 × 5 stem conv (σ_W² = 1/25, since no ReLU comes before it), max pool, block 1, max pool, blocks 2 and 3, then GroupNorm, ReLU, global average pool and a linear layer to 10 outputs. Each block is x + conv(ReLU(GN(conv(ReLU(GN(x)))))) with He-initialized convs.

The slides used 512 training digits, standardized to mean 0 and variance 1. They are the first 512 training digits labelled 0 (the batch came from a label-sorted file), so we take the same 512 here. Backward, the gradient arriving at the output has variance 1.

One printed number differs from the slide: block 1's second conv has a weight-gradient rms of 1.38 here, and 1.37 on the slide (1.3747, computed from a copy of these digits stored to 3 decimals). Everything else matches.

In [ ]:
def he_init(conv, gain=2.0):
    """Normal init with variance gain / fan_in, zero bias."""
    fan_in = conv.in_channels * conv.kernel_size[0] * conv.kernel_size[1]
    nn.init.normal_(conv.weight, 0, (gain / fan_in) ** 0.5)
    nn.init.zeros_(conv.bias)
    return conv


def network_variance(X, seed=0):
    """Forward and backward second moments at every step of Tuesday's network, plus its weight gradients."""
    torch.manual_seed(seed)
    stem = he_init(nn.Conv2d(1, 32, 5, padding=2), gain=1.0)
    blocks = [[nn.GroupNorm(1, 32), he_init(nn.Conv2d(32, 32, 3, padding=1)),
               nn.GroupNorm(1, 32), he_init(nn.Conv2d(32, 32, 3, padding=1))] for _ in range(3)]
    head_norm = nn.GroupNorm(1, 32)
    fc = nn.Linear(32, 10)
    nn.init.normal_(fc.weight, 0, (1 / 32) ** 0.5)
    nn.init.zeros_(fc.bias)

    kept = {}

    def keep(name, tensor):
        """Remember an intermediate tensor and ask autograd to keep its gradient."""
        tensor.retain_grad()
        kept[name] = tensor
        return tensor

    h = keep("x", X.clone().requires_grad_(True))
    h = keep("stem conv", stem(h))
    h = keep("X0 (after pool)", F.max_pool2d(h, 2))
    for b, (norm1, conv1, norm2, conv2) in enumerate(blocks, start=1):
        a = keep(f"block {b} norm 1", norm1(h))
        a = keep(f"block {b} ReLU 1", F.relu(a))
        a = keep(f"block {b} conv 1", conv1(a))
        a = keep(f"block {b} norm 2", norm2(a))
        a = keep(f"block {b} ReLU 2", F.relu(a))
        a = keep(f"block {b} conv 2", conv2(a))
        h = keep(f"X{b}", h + a)
        if b == 1:
            h = keep("X1 (after pool)", F.max_pool2d(h, 2))
    z = keep("head norm", head_norm(h))
    z = keep("head ReLU", F.relu(z))
    z = keep("avg pool", z.mean(dim=(2, 3)))
    out = keep("out", fc(z))
    upstream = torch.randn_like(out)                 # dL/d(out), variance 1
    (out * upstream).sum().backward()

    rows = {}
    for name, tensor in kept.items():
        values = tensor.detach()
        rows[name] = {"var": values.var().item(), "E2": values.pow(2).mean().item(), "grad E2": tensor.grad.pow(2).mean().item()}
    weights = {"stem": (stem, "stem conv"), "fc": (fc, "out")}
    for b, (norm1, conv1, norm2, conv2) in enumerate(blocks, start=1):
        weights[f"block {b} conv 1"] = (conv1, f"block {b} conv 1")
        weights[f"block {b} conv 2"] = (conv2, f"block {b} conv 2")
    weight_grads = {}
    for name, (layer, output_name) in weights.items():
        weight_grads[name] = {
            "g rms": kept[output_name].grad.pow(2).mean().sqrt().item(),
            "dW rms": layer.weight.grad.pow(2).mean().sqrt().item(),
            "|dW| / |W|": (layer.weight.grad.norm() / layer.weight.norm()).item(),
        }
    return rows, weight_grads, kept


zeros = images_train[labels_train == 0][:512]
X_batch = (zeros - zeros.mean()) / zeros.std()
rows, weight_grads, kept = network_variance(X_batch)

print(f"{'step':18s} {'σ²':>7s} {'E[·²]':>7s}   {'E[(∂L/∂·)²]':>12s}")
for name, row in rows.items():
    print(f"{name:18s} {row['var']:7.2f} {row['E2']:7.2f}   {row['grad E2'] * 1e6:10.1f} µ")
print()
for name, row in weight_grads.items():
    print(f"{name:16s} g rms {row['g rms']:.4f} → dW rms {row['dW rms']:.2f}   |dW|/|W| {row['|dW| / |W|']:.1f}")

The slides also draw each tensor's distribution. Here are the residual stream and the first norm and ReLU of block 1: the stream is right-skewed, so the ReLU after its norm keeps more than half of the second moment.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, name in zip(axes, ["X0 (after pool)", "X1", "block 1 norm 1", "block 1 ReLU 1"]):
    values = kept[name].detach().flatten().numpy()
    ax.hist(values, bins=80, range=(-3, 5), color=ORANGE, density=True)
    ax.set_title(f"{name}: σ² {values.var():.2f}")
    ax.set_xlabel("value")
    ax.set_ylabel("density")
plt.tight_layout()
plt.show()

for name in ["block 1 norm 1", "block 2 norm 1", "head norm"]:
    values = kept[name].detach().flatten()
    skew = ((values - values.mean()) ** 3).mean() / values.var() ** 1.5
    kept_by_relu = values.clamp(min=0).pow(2).mean()
    print(f"{name:15s} skew {skew:.2f}   E[ReLU(x̂)²] {kept_by_relu:.2f}  (½ for a normal x̂)")

## 5. Max pooling is a gated activation; making the gate soft

The running example is Tuesday's 7: the 183rd 7 in the MNIST training set, drawn at 10 grey levels as on Tuesday's slides. Its top-edge map is the ReLU of a 5 × 5 filter (two rows of −1, a row of 0, two rows of +1). We find its strongest 2 × 2 window V, then:

- pool it with the hard gate, V · hardmax(V);
- pool it with the soft gate, V · softmax(V / τ), across the temperature dial;
- compare ∂out/∂vⱼ for the hard gate and the soft gate at τ = 1, using ∂out/∂vⱼ = gⱼ(1 + (vⱼ − out)/τ).

The *Make the gate soft* slide computed e^v from v rounded to three decimals, so it shows e^v₃ = 4,647, e^v₄ = 9,054 and a sum of 13,794; the exact values below are 4,649, 9,055 and 13,797. The gates and the output agree.

In [ ]:
sevens = images_train[labels_train == 7]
seven = torch.round(sevens[182, 0] * 9) / 9               # Tuesday's 7, at 10 grey levels

top_edge_filter = torch.tensor([-1.0] * 10 + [0.0] * 5 + [1.0] * 10).view(1, 1, 5, 5)
edge_map = F.relu(F.conv2d(seven.view(1, 1, 28, 28), top_edge_filter))[0, 0].double()   # 24 x 24
# every value is a multiple of 1/9 (10 grey levels); rounding to that grid removes float noise,
# so when two windows tie for the largest value, the first one (top to bottom, left to right) wins
edge_map = torch.round(edge_map * 9) / 9

windows_of_map = edge_map.view(12, 2, 12, 2).permute(0, 2, 1, 3).reshape(12, 12, 4)
row, col = divmod(windows_of_map.amax(dim=-1).argmax().item(), 12)
V = windows_of_map[row, col]
print(f"strongest window: row {row}, column {col}, V = {V.numpy().round(2)}")

hard_gate = F.one_hot(V.argmax(), 4).double()
print(f"max pool: V · hardmax(V) = {V @ hard_gate:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(8, 3.8))
ax[0].imshow(seven, cmap="gray_r")
ax[0].set_title("Tuesday's 7")
ax[1].imshow(edge_map, cmap="Blues")
ax[1].add_patch(plt.Rectangle((2 * col - 0.5, 2 * row - 0.5), 2, 2, fill=False, edgecolor=ORANGE, linewidth=2))
ax[1].set_title("Top-edge map; strongest window in orange")
for a in ax:
    a.set_xlabel("column")
    a.set_ylabel("row")
plt.tight_layout()
plt.show()

In [ ]:
def soft_pool(V, tau):
    """V · softmax(V / tau): the gate and the pooled output."""
    gate = torch.softmax(V / tau, dim=0)
    return gate, (V * gate).sum()


print("e^v:", ", ".join(f"{value:,.0f}" for value in torch.exp(V).tolist()), f"   sum {torch.exp(V).sum().item():,.0f}")
print()
print(f"{'τ':>6s}   gates g₁ … g₄                    out")
for tau in [0.01, 0.5, 1.0, 2.0, 100.0]:
    gate, out = soft_pool(V, tau)
    print(f"{tau:6g}   {np.array2string(gate.numpy().round(3) + 0.0, precision=3, floatmode='fixed')}   {out:.2f}")
print(f"τ → ∞ (average pooling): {V.mean():.2f}")

tau = 1.0
gate, out = soft_pool(V, tau)
soft_gradient = gate * (1 + (V - out) / tau)
V_check = V.clone().requires_grad_(True)
soft_pool(V_check, tau)[1].backward()
print()
print("∂out/∂v, hard max      :", hard_gate.numpy())
print("∂out/∂v, softmax, τ = 1:", soft_gradient.numpy().round(3))
print("autograd agrees        :", torch.allclose(V_check.grad, soft_gradient))

## 6. Does it matter on MNIST?

We swap both pools of Tuesday's network for each of six pools, train on full MNIST and record test accuracy. Every pool turns the 4 positions of a 2 × 2 window into 1:

| pool | gate over the 4 positions | value |
|---|---|---|
| max | hardmax(V) | V |
| average | ¼ each | V |
| SoftPool | softmax(V), per channel | V |
| SoftPool, learned τ | softmax(V / τ), one τ per channel | V |
| attention pooling | softmax(Xwᵀ): one learned score per position, shared by all channels | XVᵀ |
| 2 × 2 stride-2 conv | fixed learned weights, no gate | X |

Every run uses Adam at learning rate 10⁻³, batch 128 and 6 epochs, with 3 seeds per pool (the seed sets both the initialization and the batch order). Training on a GPU is not bit-for-bit repeatable, so your accuracies will differ from the slide's in the second decimal; the pattern should not.

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = "/content/drive/MyDrive/ai4200_pooling_gates"
else:
    CACHE_DIR = "pooling_gates_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = 1 if QUICK else 6
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
POOLS = ["max", "average", "softpool", "softpool_learned_tau", "attention", "stride2_conv"]
POOL_COLORS = {"max": INK, "average": GREY, "softpool": ORANGE, "softpool_learned_tau": "#E39A5F",
               "attention": BLUE, "stride2_conv": "#7A8BB8"}

# the usual MNIST standardization, held on the CPU; one batch at a time goes to the device
X_train = (images_train - 0.1307) / 0.3081
X_test = (images_test - 0.1307) / 0.3081
y_train, y_test = labels_train, labels_test
if QUICK:
    X_train, y_train = X_train[:2000], y_train[:2000]
    X_test, y_test = X_test[:1000], y_test[:1000]
print("train", tuple(X_train.shape), " test", tuple(X_test.shape))

In [ ]:
def windows(X):
    """Reshape N x C x H x W into N x C x H/2 x W/2 x 4: the 4 positions of each 2 x 2 window."""
    N, C, H, W = X.shape
    X = X.reshape(N, C, H // 2, 2, W // 2, 2)
    return X.permute(0, 1, 2, 4, 3, 5).reshape(N, C, H // 2, W // 2, 4)


class MaxPool(nn.Module):
    def forward(self, X):
        V = windows(X)
        gate = F.one_hot(V.argmax(dim=-1), 4).to(V.dtype)   # hardmax(V)
        return (V * gate).sum(dim=-1)                      # V · hardmax(V)


class AveragePool(nn.Module):
    def forward(self, X):
        return windows(X).mean(dim=-1)


class SoftPool(nn.Module):
    """V · softmax(V / tau) in each window; tau is fixed at 1 or learned per channel."""

    def __init__(self, channels, learn_tau=False):
        super().__init__()
        # learn log(tau) so tau stays positive; start at tau = 1 (plain SoftPool)
        self.log_tau = nn.Parameter(torch.zeros(channels), requires_grad=learn_tau)

    def forward(self, X):
        V = windows(X)
        tau = self.log_tau.exp().view(1, -1, 1, 1, 1)
        gate = torch.softmax(V / tau, dim=-1)
        return (V * gate).sum(dim=-1)


class AttentionPool(nn.Module):
    """softmax(X w^T)^T · X V^T in each window: a learned score w, and a value projection V."""

    def __init__(self, channels):
        super().__init__()
        self.w = nn.Linear(channels, 1, bias=False)          # gate path: one score per position
        self.V = nn.Linear(channels, channels, bias=False)   # value path
        nn.init.normal_(self.V.weight, 0, (1 / channels) ** 0.5)

    def forward(self, X):
        Xw = windows(X).permute(0, 2, 3, 4, 1)               # N x H/2 x W/2 x 4 positions x C
        gate = torch.softmax(self.w(Xw), dim=-2)             # N x H/2 x W/2 x 4 x 1
        out = (gate * self.V(Xw)).sum(dim=-2)                # N x H/2 x W/2 x C
        return out.permute(0, 3, 1, 2)


def stride2_conv(channels):
    """A 2 x 2, stride-2 conv: a learned reduction with fixed weights per position."""
    conv = nn.Conv2d(channels, channels, 2, stride=2)
    nn.init.normal_(conv.weight, 0, (1 / (4 * channels)) ** 0.5)
    nn.init.zeros_(conv.bias)
    return conv


def make_pool(name, channels):
    """Build one pooling layer by name."""
    if name == "max":
        return MaxPool()
    if name == "average":
        return AveragePool()
    if name == "softpool":
        return SoftPool(channels)
    if name == "softpool_learned_tau":
        return SoftPool(channels, learn_tau=True)
    if name == "attention":
        return AttentionPool(channels)
    if name == "stride2_conv":
        return stride2_conv(channels)
    raise ValueError(name)


X_check = torch.randn(2, 3, 8, 8)
print("MaxPool matches F.max_pool2d:", torch.allclose(MaxPool()(X_check), F.max_pool2d(X_check, 2)))
print("AveragePool matches F.avg_pool2d:", torch.allclose(AveragePool()(X_check), F.avg_pool2d(X_check, 2)))

In [ ]:
class Block(nn.Module):
    """x + conv(ReLU(GN(conv(ReLU(GN(x)))))), He init after each ReLU."""

    def __init__(self, channels):
        super().__init__()
        self.norm1 = nn.GroupNorm(1, channels)
        self.conv1 = he_init(nn.Conv2d(channels, channels, 3, padding=1))
        self.norm2 = nn.GroupNorm(1, channels)
        self.conv2 = he_init(nn.Conv2d(channels, channels, 3, padding=1))

    def forward(self, X):
        h = self.conv1(F.relu(self.norm1(X)))
        h = self.conv2(F.relu(self.norm2(h)))
        return X + h


class Net(nn.Module):
    """Tuesday's network, with both pools chosen by name."""

    def __init__(self, pool_name, channels=32):
        super().__init__()
        self.stem = he_init(nn.Conv2d(1, channels, 5, padding=2), gain=1.0)   # no ReLU before it
        self.pool1 = make_pool(pool_name, channels)
        self.block1 = Block(channels)
        self.pool2 = make_pool(pool_name, channels)
        self.block2 = Block(channels)
        self.block3 = Block(channels)
        self.head_norm = nn.GroupNorm(1, channels)
        self.fc = nn.Linear(channels, 10)
        nn.init.normal_(self.fc.weight, 0, (1 / channels) ** 0.5)
        nn.init.zeros_(self.fc.bias)

    def forward(self, X, keep=None):
        h = self.stem(X)
        if keep is not None:
            keep["pool1_in"] = h
        h = self.pool1(h)
        h = self.block1(h)
        if keep is not None:
            keep["pool2_in"] = h
        h = self.pool2(h)
        h = self.block3(self.block2(h))
        h = F.relu(self.head_norm(h)).mean(dim=(2, 3))
        return self.fc(h)


def gradient_share(pool_name, seed=0, n_examples=512):
    """Fraction of each pool's inputs that receive a nonzero gradient, in an untrained network."""
    torch.manual_seed(seed)
    model = Net(pool_name).to(device)
    xb, yb = X_train[:n_examples].to(device), y_train[:n_examples].to(device)
    keep = {}
    y_hat = model(xb, keep=keep)
    for tensor in keep.values():
        tensor.retain_grad()
    F.cross_entropy(y_hat, yb).backward()
    return [(keep[k].grad != 0).float().mean().item() for k in ("pool1_in", "pool2_in")]


for name in POOLS:
    share = gradient_share(name)
    n_params = sum(p.numel() for p in Net(name).parameters() if p.requires_grad)
    print(f"{name:22s} {n_params:7,d} parameters   gets a gradient: pool 1 {share[0]:.0%}, pool 2 {share[1]:.0%}")

In [ ]:
def evaluate(model, X, y):
    """Mean loss and accuracy over a dataset, in batches."""
    model.eval()
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for start in range(0, len(X), 1000):
            xb, yb = X[start:start + 1000].to(device), y[start:start + 1000].to(device)
            y_hat = model(xb)
            total_loss += F.cross_entropy(y_hat, yb, reduction="sum").item()
            correct += (y_hat.argmax(dim=1) == yb).sum().item()
    return total_loss / len(X), correct / len(X)


def train(pool_name, seed):
    """Train one network; return per-epoch test metrics (and the learned temperatures, if any)."""
    torch.manual_seed(seed)
    model = Net(pool_name).to(device)
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
    generator = torch.Generator().manual_seed(seed)
    history = {"test_loss": [], "test_acc": [], "seconds": []}
    for epoch in range(EPOCHS):
        model.train()
        start_time = time.time()
        order = torch.randperm(len(X_train), generator=generator)
        for start in range(0, len(X_train), BATCH_SIZE):
            idx = order[start:start + BATCH_SIZE]
            xb, yb = X_train[idx].to(device), y_train[idx].to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            optimizer.step()
        history["seconds"].append(time.time() - start_time)
        test_loss, test_acc = evaluate(model, X_test, y_test)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)
        print(f"  {pool_name} seed {seed} epoch {epoch + 1}: test accuracy {test_acc:.4f}")
    if pool_name == "softpool_learned_tau":
        history["tau_pool1"] = model.pool1.log_tau.exp().tolist()
        history["tau_pool2"] = model.pool2.log_tau.exp().tolist()
    return history


tag = "quick" if QUICK else "full"
results = {}
for name in POOLS:
    for seed in SEEDS:
        path = os.path.join(CACHE_DIR, f"{tag}_{name}_seed{seed}.json")
        if os.path.exists(path):
            with open(path) as f:
                results[(name, seed)] = json.load(f)
            continue
        results[(name, seed)] = train(name, seed)
        with open(path, "w") as f:
            json.dump(results[(name, seed)], f)

### Results

One dot per seed and the mean as a bar, as on the slide. Then the learned temperatures: each channel of the learned-τ SoftPool starts at τ = 1, and a smaller τ moves it toward max pooling, a larger one toward average pooling.

In [ ]:
final_acc = {name: 100 * np.array([results[(name, seed)]["test_acc"][-1] for seed in SEEDS]) for name in POOLS}
for name in POOLS:
    print(f"{name:22s} {final_acc[name].mean():6.2f} % (seeds: {np.array2string(final_acc[name], precision=2)})")

fig, ax = plt.subplots(figsize=(8, 3.6))
for i, name in enumerate(POOLS):
    ax.barh(i, final_acc[name].mean() - 97.5, left=97.5, color=POOL_COLORS[name], alpha=0.35)
    ax.scatter(final_acc[name], [i] * len(SEEDS), color=POOL_COLORS[name], zorder=3)
ax.set_yticks(range(len(POOLS)))
ax.set_yticklabels(POOLS)
ax.invert_yaxis()
ax.set_xlabel(f"test accuracy after epoch {EPOCHS} (%)")
ax.set_title("Swapping both pools in Tuesday's network")
plt.tight_layout()
plt.show()

taus = np.concatenate([results[("softpool_learned_tau", seed)][key] for seed in SEEDS for key in ("tau_pool1", "tau_pool2")])
print(f"learned τ: from {taus.min():.2f} to {taus.max():.2f} (median {np.median(taus):.2f}), starting from 1")
plt.figure(figsize=(6, 3.2))
plt.hist(np.log10(taus), bins=20, color=ORANGE, edgecolor="white")
plt.axvline(0, color=INK, linestyle="--", label="start, τ = 1")
plt.title("Where the learned temperatures end up")
plt.xlabel("log₁₀ τ   (← toward max, toward average →)")
plt.ylabel("channels")
plt.legend()
plt.show()